In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Bronze Layer — Step 1: Verify Raw Source File

source_file = "/Volumes/workspace/default/raw_sop_source/Instance data.xlsx"

print("Raw S&OP source:")
print(source_file)
print()

files = dbutils.fs.ls("/Volumes/workspace/default/raw_sop_source/")

for file in files:
    print(
        f"Name: {file.name} | "
        f"Size: {file.size:,} bytes | "
        f"Path: {file.path}"
    )

Raw S&OP source:
/Volumes/workspace/default/raw_sop_source/Instance data.xlsx

Name: Instance data.xlsx | Size: 124,741 bytes | Path: dbfs:/Volumes/workspace/default/raw_sop_source/Instance data.xlsx


In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Bronze Layer — Step 2: Inspect Excel Workbook Structure
# Read workbook metadata only — no transformations

import pandas as pd

excel_file = pd.ExcelFile(source_file)

sheet_names = excel_file.sheet_names

print(f"Workbook: Instance data.xlsx")
print(f"Number of worksheets: {len(sheet_names)}")
print()
print("Worksheets found:")

for i, sheet in enumerate(sheet_names, start=1):
    print(f"{i}. {sheet}")

Workbook: Instance data.xlsx
Number of worksheets: 7

Worksheets found:
1. Gerenal parameters
2. Products
3. Demand
4. Production
5. Packing
6. Semi-finished prod acquisition
7. Raw materials acquisition


In [0]:
# Bronze Layer — Step 3: Profile Raw Worksheet Dimensions
# No cleaning or transformations are performed in this step

print("RAW WORKBOOK PROFILE")
print("=" * 70)

sheet_profile = {}

for sheet in sheet_names:
    df_raw = pd.read_excel(
        source_file,
        sheet_name=sheet,
        header=None
    )

    rows, columns = df_raw.shape

    sheet_profile[sheet] = {
        "rows": rows,
        "columns": columns
    }

    print(
        f"{sheet:<35} "
        f"Rows: {rows:<5} "
        f"Columns: {columns}"
    )

RAW WORKBOOK PROFILE
Gerenal parameters                  Rows: 9     Columns: 8
Products                            Rows: 1027  Columns: 23
Demand                              Rows: 247   Columns: 14
Production                          Rows: 735   Columns: 18
Packing                             Rows: 190   Columns: 11
Semi-finished prod acquisition      Rows: 41    Columns: 3
Raw materials acquisition           Rows: 5     Columns: 4


In [0]:
# Bronze Layer — Step 4: Inspect Raw Worksheet Contents
# Use pandas text output to preserve mixed Excel data types.
# No cleaning, header assignment, or transformations.

for sheet in sheet_names:

    print("\n" + "=" * 100)
    print(f"WORKSHEET: {sheet}")
    print("=" * 100)

    df_raw = pd.read_excel(
        source_file,
        sheet_name=sheet,
        header=None
    )

    print(df_raw.head(15).to_string(index=True, header=True))


WORKSHEET: Gerenal parameters
                                                         0           1   2   3       4                    5                   6               7
0                                        Number of periods    8.000000 NaN NaN  Period  Macroperiod (month)  Last day of period  Number of days
1                                     Total number of days  101.000000 NaN NaN       1                    1                   7               7
2  Penalty (price depreciation rate) per each day of dealy    0.011111 NaN NaN       2                    1                  15               8
3      Additional production cost (%) per unit of overtime    0.250000 NaN NaN       3                    2                  21               6
4         Additional packing cost (%) per unit of overtime    0.250000 NaN NaN       4                    2                  28               7
5                           Raw materials Holding cost (%)    0.100000 NaN NaN       5                   

In [0]:
# Bronze Layer — Step 5: Structural Audit of Raw Worksheets
# Identify populated rows/columns without transforming source data.

for sheet in sheet_names:
    df_raw = pd.read_excel(
        source_file,
        sheet_name=sheet,
        header=None
    )

    print("\n" + "=" * 100)
    print(f"WORKSHEET: {sheet}")
    print(f"RAW DIMENSIONS: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
    print("=" * 100)

    # Count populated cells by column
    print("\nNON-NULL CELLS BY COLUMN:")
    for col in df_raw.columns:
        count = int(df_raw[col].notna().sum())
        print(f"Column {col}: {count}")

    # Show first 10 rows with explicit row/column coordinates
    print("\nFIRST 10 ROWS:")
    for row_idx in range(min(10, len(df_raw))):
        values = []

        for col_idx in df_raw.columns:
            value = df_raw.iloc[row_idx, col_idx]

            if pd.notna(value):
                values.append(f"C{col_idx}={repr(value)}")

        print(f"R{row_idx}: " + " | ".join(values))


WORKSHEET: Gerenal parameters
RAW DIMENSIONS: 9 rows x 8 columns

NON-NULL CELLS BY COLUMN:
Column 0: 7
Column 1: 7
Column 2: 0
Column 3: 0
Column 4: 9
Column 5: 9
Column 6: 9
Column 7: 9

FIRST 10 ROWS:
R0: C0='Number of periods' | C1=np.float64(8.0) | C4='Period' | C5='Macroperiod (month)' | C6='Last day of period' | C7='Number of days'
R1: C0='Total number of days' | C1=np.float64(101.0) | C4=1 | C5=1 | C6=7 | C7=7
R2: C0='Penalty (price depreciation rate) per each day of dealy' | C1=np.float64(0.011111111111111112) | C4=2 | C5=1 | C6=15 | C7=8
R3: C0='Additional production cost (%) per unit of overtime' | C1=np.float64(0.25) | C4=3 | C5=2 | C6=21 | C7=6
R4: C0='Additional packing cost (%) per unit of overtime' | C1=np.float64(0.25) | C4=4 | C5=2 | C6=28 | C7=7
R5: C0='Raw materials Holding cost (%)' | C1=np.float64(0.1) | C4=5 | C5=2 | C6=35 | C7=7
R6: C0='Products Holding cost (%)' | C1=np.float64(0.1) | C4=6 | C5=2 | C6=44 | C7=9
R7: C4=7 | C5=3 | C6=72 | C7=28
R8: C4=8 | C5=4 |

In [0]:
# Bronze Layer — Step 6: Define Verified Raw Source Blocks
# These boundaries come directly from the Step 5 structural audit.
# No business-data transformation is performed here.

raw_blocks = [
    # General parameters
     ("Gerenal parameters", "planning_parameters", [0, 1]),
    ("Gerenal parameters", "planning_periods", [4, 5, 6, 7]),

    # Products
    ("Products", "finished_product_type", [0, 1]),
    ("Products", "finished_product_economics", [3, 4, 5, 6]),
    ("Products", "semi_finished_product_cost", [8, 9]),
    ("Products", "product_inventory", [11, 12]),
    ("Products", "product_raw_material_requirement", [14, 15, 16, 17, 18]),
    ("Products", "product_successor", [20, 21, 22]),

    # Demand
    ("Demand", "customer_order", [0, 1, 2, 3, 4]),
    ("Demand", "demand_plan", [6, 7, 8, 9]),
    ("Demand", "attainment_parameter", [11, 12, 13]),

    # Production
    ("Production", "product_lot_size", [0, 1]),
    ("Production", "product_production_alternative", [3, 4, 5]),
    ("Production", "alternative_machine", [7, 8, 9]),
    ("Production", "machine", [11, 12]),
    ("Production", "machine_capacity", [14, 15, 16, 17]),

    # Packing
    ("Packing", "packing_machine", [0, 1]),
    ("Packing", "product_packing_machine", [3, 4, 5]),
    ("Packing", "packing_capacity", [7, 8, 9, 10]),

    # Acquisition
    ("Semi-finished prod acquisition", "semi_finished_acquisition", [0, 1, 2]),
    ("Raw materials acquisition", "raw_material", [0, 1, 2, 3]),
]

print("VERIFIED RAW SOURCE BLOCK INVENTORY")
print("=" * 90)
print(f"Total logical blocks: {len(raw_blocks)}")
print()

for sheet, block, columns in raw_blocks:
    print(
        f"{sheet:<35} | "
        f"{block:<35} | "
        f"Columns: {columns}"
    )

VERIFIED RAW SOURCE BLOCK INVENTORY
Total logical blocks: 21

Gerenal parameters                  | planning_parameters                 | Columns: [0, 1]
Gerenal parameters                  | planning_periods                    | Columns: [4, 5, 6, 7]
Products                            | finished_product_type               | Columns: [0, 1]
Products                            | finished_product_economics          | Columns: [3, 4, 5, 6]
Products                            | semi_finished_product_cost          | Columns: [8, 9]
Products                            | product_inventory                   | Columns: [11, 12]
Products                            | product_raw_material_requirement    | Columns: [14, 15, 16, 17, 18]
Products                            | product_successor                   | Columns: [20, 21, 22]
Demand                              | customer_order                      | Columns: [0, 1, 2, 3, 4]
Demand                              | demand_plan                  

In [0]:
# Bronze Layer — Step 7: Extract Verified Raw Source Blocks
# Extract each verified logical block while correctly handling
# the headerless planning_parameters block.

print("BRONZE RAW BLOCK RECONCILIATION")
print("=" * 110)

bronze_blocks = {}

for sheet_name, block_name, column_positions in raw_blocks:

    # Read the worksheet exactly as stored in Excel.
    df_raw = pd.read_excel(
        source_file,
        sheet_name=sheet_name,
        header=None
    )

    # Select only the verified columns belonging to this logical block.
    df_block = df_raw.iloc[:, column_positions].copy()

    # ----------------------------------------------------------
    # Special case: planning_parameters has NO source header row.
    # Every populated row is a business record.
    # ----------------------------------------------------------
    if block_name == "planning_parameters":

        df_block = df_block.dropna(how="all")

        df_block.columns = [
            "parameter_name",
            "parameter_value"
        ]

    # ----------------------------------------------------------
    # All other verified blocks use their first selected row
    # as the source header.
    # ----------------------------------------------------------
    else:

        source_headers = df_block.iloc[0].tolist()

        # Remove the source header row.
        df_block = df_block.iloc[1:].copy()

        # Remove completely empty rows only.
        df_block = df_block.dropna(how="all")

        # Preserve the source column names.
        df_block.columns = source_headers

    # Reset index after extraction.
    df_block = df_block.reset_index(drop=True)

    # Store this logical block.
    bronze_blocks[block_name] = df_block

    # IMPORTANT: this print remains INSIDE the for loop.
    print(
        f"{block_name:<40} | "
        f"Rows: {len(df_block):<5} | "
        f"Columns: {len(df_block.columns)}"
    )

print("=" * 110)
print(f"Logical blocks extracted: {len(bronze_blocks)}")

BRONZE RAW BLOCK RECONCILIATION
planning_parameters                      | Rows: 7     | Columns: 2
planning_periods                         | Rows: 8     | Columns: 4
finished_product_type                    | Rows: 67    | Columns: 2
finished_product_economics               | Rows: 67    | Columns: 4
semi_finished_product_cost               | Rows: 225   | Columns: 2
product_inventory                        | Rows: 292   | Columns: 2
product_raw_material_requirement         | Rows: 292   | Columns: 5
product_successor                        | Rows: 1026  | Columns: 3
customer_order                           | Rows: 61    | Columns: 5
demand_plan                              | Rows: 246   | Columns: 4
attainment_parameter                     | Rows: 3     | Columns: 3
product_lot_size                         | Rows: 292   | Columns: 2
product_production_alternative           | Rows: 592   | Columns: 3
alternative_machine                      | Rows: 734   | Columns: 3
machine         

In [0]:
# Bronze Layer — Step 8: Validate Extracted Raw Block Record Counts
# Expected counts are derived from the verified Step 5/Step 7 source audit.
# This validation checks extraction completeness before persistence.

expected_row_counts = {
    "planning_parameters": 7,
    "planning_periods": 8,
    "finished_product_type": 67,
    "finished_product_economics": 67,
    "semi_finished_product_cost": 225,
    "product_inventory": 292,
    "product_raw_material_requirement": 292,
    "product_successor": 1026,
    "customer_order": 61,
    "demand_plan": 246,
    "attainment_parameter": 3,
    "product_lot_size": 292,
    "product_production_alternative": 592,
    "alternative_machine": 734,
    "machine": 39,
    "machine_capacity": 312,
    "packing_machine": 7,
    "product_packing_machine": 189,
    "packing_capacity": 56,
    "semi_finished_acquisition": 40,
    "raw_material": 4
}

validation_results = []

print("BRONZE EXTRACTION VALIDATION")
print("=" * 100)

for block_name, expected_rows in expected_row_counts.items():

    actual_rows = len(bronze_blocks[block_name])
    status = "PASS" if actual_rows == expected_rows else "FAIL"

    validation_results.append({
        "block_name": block_name,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "status": status
    })

    print(
        f"{block_name:<40} | "
        f"Expected: {expected_rows:<5} | "
        f"Actual: {actual_rows:<5} | "
        f"{status}"
    )

print("=" * 100)

failed_blocks = [
    result for result in validation_results
    if result["status"] == "FAIL"
]

if failed_blocks:
    raise ValueError(
        f"Bronze extraction validation failed for "
        f"{len(failed_blocks)} block(s)."
    )

print(
    f"VALIDATION PASSED: "
    f"{len(validation_results)} of {len(validation_results)} "
    f"raw source blocks reconciled successfully."
)

BRONZE EXTRACTION VALIDATION
planning_parameters                      | Expected: 7     | Actual: 7     | PASS
planning_periods                         | Expected: 8     | Actual: 8     | PASS
finished_product_type                    | Expected: 67    | Actual: 67    | PASS
finished_product_economics               | Expected: 67    | Actual: 67    | PASS
semi_finished_product_cost               | Expected: 225   | Actual: 225   | PASS
product_inventory                        | Expected: 292   | Actual: 292   | PASS
product_raw_material_requirement         | Expected: 292   | Actual: 292   | PASS
product_successor                        | Expected: 1026  | Actual: 1026  | PASS
customer_order                           | Expected: 61    | Actual: 61    | PASS
demand_plan                              | Expected: 246   | Actual: 246   | PASS
attainment_parameter                     | Expected: 3     | Actual: 3     | PASS
product_lot_size                         | Expected: 292   | Actual: 

In [0]:
# Bronze Layer — Step 9: Audit Source Column Names
# Inspect exact source headers before preparing Delta-compatible names.
# No source values are modified.

print("BRONZE SOURCE COLUMN AUDIT")
print("=" * 120)

for block_name, df_block in bronze_blocks.items():

    print(f"\nBLOCK: {block_name}")
    print("-" * 120)

    for position, column_name in enumerate(df_block.columns, start=1):
        print(
            f"{position:<3} | "
            f"{repr(column_name)}"
        )

print("\n" + "=" * 120)
print(f"Blocks audited: {len(bronze_blocks)}")
print(
    f"Total source columns audited: "
    f"{sum(len(df.columns) for df in bronze_blocks.values())}"
)

BRONZE SOURCE COLUMN AUDIT

BLOCK: planning_parameters
------------------------------------------------------------------------------------------------------------------------
1   | 'parameter_name'
2   | 'parameter_value'

BLOCK: planning_periods
------------------------------------------------------------------------------------------------------------------------
1   | 'Period'
2   | 'Macroperiod (month)'
3   | 'Last day of period'
4   | 'Number of days'

BLOCK: finished_product_type
------------------------------------------------------------------------------------------------------------------------
1   | 'Finished product'
2   | 'MTS/MTO'

BLOCK: finished_product_economics
------------------------------------------------------------------------------------------------------------------------
1   | 'Finished product'
2   | 'Industrial cost (€/m)'
3   | 'Margin (%)'
4   | 'Average sales price (€/m)'

BLOCK: semi_finished_product_cost
-----------------------------------------------

In [0]:
# Bronze Layer — Step 10: Final Raw Block Validation
# Revalidate all extracted blocks after correcting the headerless
# planning_parameters source structure.



print("FINAL BRONZE RAW BLOCK VALIDATION")
print("=" * 100)

final_validation_results = []

for block_name, expected_rows in expected_row_counts.items():

    actual_rows = len(bronze_blocks[block_name])
    status = "PASS" if actual_rows == expected_rows else "FAIL"

    final_validation_results.append({
        "block_name": block_name,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "status": status
    })

    print(
        f"{block_name:<40} | "
        f"Expected: {expected_rows:<5} | "
        f"Actual: {actual_rows:<5} | "
        f"{status}"
    )

print("=" * 100)

failed_blocks = [
    result for result in final_validation_results
    if result["status"] == "FAIL"
]

if failed_blocks:
    raise ValueError(
        f"Final Bronze validation failed for "
        f"{len(failed_blocks)} block(s)."
    )

print(
    f"FINAL VALIDATION PASSED: "
    f"{len(final_validation_results)} of "
    f"{len(final_validation_results)} raw source blocks "
    f"reconciled successfully."
)

FINAL BRONZE RAW BLOCK VALIDATION
planning_parameters                      | Expected: 7     | Actual: 7     | PASS
planning_periods                         | Expected: 8     | Actual: 8     | PASS
finished_product_type                    | Expected: 67    | Actual: 67    | PASS
finished_product_economics               | Expected: 67    | Actual: 67    | PASS
semi_finished_product_cost               | Expected: 225   | Actual: 225   | PASS
product_inventory                        | Expected: 292   | Actual: 292   | PASS
product_raw_material_requirement         | Expected: 292   | Actual: 292   | PASS
product_successor                        | Expected: 1026  | Actual: 1026  | PASS
customer_order                           | Expected: 61    | Actual: 61    | PASS
demand_plan                              | Expected: 246   | Actual: 246   | PASS
attainment_parameter                     | Expected: 3     | Actual: 3     | PASS
product_lot_size                         | Expected: 292   | Act

In [0]:
# Bronze Layer — Step 11: Create Delta-Safe Technical Column Names
# Standardize column names for Bronze Delta persistence while preserving
# source values and source-to-target column lineage.

import re

def to_delta_safe_column_name(column_name):
    """
    Convert a source column name to a lowercase snake_case technical name.
    Source data values are not modified.
    """

    name = str(column_name).strip().lower()

    # Preserve meaning of selected symbols before removing punctuation
    name = name.replace("%", " pct ")
    name = name.replace("€", " eur ")

    # Replace separators/punctuation with underscores
    name = re.sub(r"[^a-z0-9]+", "_", name)

    # Remove leading/trailing underscores and collapse duplicates
    name = re.sub(r"_+", "_", name).strip("_")

    return name


bronze_delta_blocks = {}
column_lineage = []

print("BRONZE DELTA COLUMN STANDARDIZATION")
print("=" * 120)

for block_name, df_block in bronze_blocks.items():

    df_delta = df_block.copy()

    original_columns = list(df_delta.columns)
    technical_columns = [
        to_delta_safe_column_name(col)
        for col in original_columns
    ]

    # Prevent ambiguous schemas
    if len(technical_columns) != len(set(technical_columns)):
        raise ValueError(
            f"Duplicate technical column names detected "
            f"in block: {block_name}"
        )

    df_delta.columns = technical_columns

    bronze_delta_blocks[block_name] = df_delta

    print(f"\nBLOCK: {block_name}")
    print("-" * 120)

    for source_col, target_col in zip(
        original_columns,
        technical_columns
    ):
        print(
            f"{repr(source_col):<70} -> {target_col}"
        )

        column_lineage.append({
            "block_name": block_name,
            "source_column": str(source_col),
            "bronze_column": target_col
        })

print("\n" + "=" * 120)
print(f"Blocks standardized: {len(bronze_delta_blocks)}")
print(f"Column mappings created: {len(column_lineage)}")

BRONZE DELTA COLUMN STANDARDIZATION

BLOCK: planning_parameters
------------------------------------------------------------------------------------------------------------------------
'parameter_name'                                                       -> parameter_name
'parameter_value'                                                      -> parameter_value

BLOCK: planning_periods
------------------------------------------------------------------------------------------------------------------------
'Period'                                                               -> period
'Macroperiod (month)'                                                  -> macroperiod_month
'Last day of period'                                                   -> last_day_of_period
'Number of days'                                                       -> number_of_days

BLOCK: finished_product_type
---------------------------------------------------------------------------------------------------------

In [0]:
# Bronze Layer — Step 12: Validate Delta-Safe Schema Preparation
# Confirm that technical column standardization changed schema names only.
# Record counts and column counts must remain unchanged.

print("BRONZE DELTA SCHEMA VALIDATION")
print("=" * 110)

schema_validation_results = []

for block_name in bronze_blocks:

    source_rows = len(bronze_blocks[block_name])
    delta_rows = len(bronze_delta_blocks[block_name])

    source_columns = len(bronze_blocks[block_name].columns)
    delta_columns = len(bronze_delta_blocks[block_name].columns)

    row_status = "PASS" if source_rows == delta_rows else "FAIL"
    column_status = "PASS" if source_columns == delta_columns else "FAIL"

    overall_status = (
        "PASS"
        if row_status == "PASS" and column_status == "PASS"
        else "FAIL"
    )

    schema_validation_results.append({
        "block_name": block_name,
        "source_rows": source_rows,
        "delta_rows": delta_rows,
        "source_columns": source_columns,
        "delta_columns": delta_columns,
        "status": overall_status
    })

    print(
        f"{block_name:<40} | "
        f"Rows: {source_rows} -> {delta_rows} | "
        f"Columns: {source_columns} -> {delta_columns} | "
        f"{overall_status}"
    )

print("=" * 110)

failed_schema_checks = [
    result for result in schema_validation_results
    if result["status"] == "FAIL"
]

if failed_schema_checks:
    raise ValueError(
        f"Bronze schema validation failed for "
        f"{len(failed_schema_checks)} block(s)."
    )

print(
    f"SCHEMA VALIDATION PASSED: "
    f"{len(schema_validation_results)} of "
    f"{len(schema_validation_results)} blocks preserved "
    f"row and column counts."
)

BRONZE DELTA SCHEMA VALIDATION
planning_parameters                      | Rows: 7 -> 7 | Columns: 2 -> 2 | PASS
planning_periods                         | Rows: 8 -> 8 | Columns: 4 -> 4 | PASS
finished_product_type                    | Rows: 67 -> 67 | Columns: 2 -> 2 | PASS
finished_product_economics               | Rows: 67 -> 67 | Columns: 4 -> 4 | PASS
semi_finished_product_cost               | Rows: 225 -> 225 | Columns: 2 -> 2 | PASS
product_inventory                        | Rows: 292 -> 292 | Columns: 2 -> 2 | PASS
product_raw_material_requirement         | Rows: 292 -> 292 | Columns: 5 -> 5 | PASS
product_successor                        | Rows: 1026 -> 1026 | Columns: 3 -> 3 | PASS
customer_order                           | Rows: 61 -> 61 | Columns: 5 -> 5 | PASS
demand_plan                              | Rows: 246 -> 246 | Columns: 4 -> 4 | PASS
attainment_parameter                     | Rows: 3 -> 3 | Columns: 3 -> 3 | PASS
product_lot_size                         | Rows: 2

In [0]:
# Bronze Layer — Step 13: Create Persistent Bronze Schema
# Create a dedicated Unity Catalog schema for Bronze Delta tables.

bronze_catalog = "workspace"
bronze_schema = "bronze_sop"

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {bronze_catalog}.{bronze_schema}
    """
)

print("BRONZE SCHEMA READY")
print("=" * 80)
print(f"Catalog: {bronze_catalog}")
print(f"Schema:  {bronze_schema}")
print(f"Target:  {bronze_catalog}.{bronze_schema}")

BRONZE SCHEMA READY
Catalog: workspace
Schema:  bronze_sop
Target:  workspace.bronze_sop


In [0]:
# Bronze Layer — Step 14: Audit Inferred Data Types
# Inspect pandas-inferred data types before Bronze Delta persistence.
# No data types or source values are changed in this step.

print("BRONZE INFERRED DATA TYPE AUDIT")
print("=" * 120)

total_columns = 0

for block_name, df_block in bronze_delta_blocks.items():

    print(f"\nBLOCK: {block_name}")
    print("-" * 120)

    for column_name, dtype in df_block.dtypes.items():
        print(
            f"{column_name:<70} | {str(dtype)}"
        )
        total_columns += 1

print("\n" + "=" * 120)
print(f"Blocks audited: {len(bronze_delta_blocks)}")
print(f"Columns audited: {total_columns}")

BRONZE INFERRED DATA TYPE AUDIT

BLOCK: planning_parameters
------------------------------------------------------------------------------------------------------------------------
parameter_name                                                         | object
parameter_value                                                        | float64

BLOCK: planning_periods
------------------------------------------------------------------------------------------------------------------------
period                                                                 | object
macroperiod_month                                                      | object
last_day_of_period                                                     | object
number_of_days                                                         | object

BLOCK: finished_product_type
------------------------------------------------------------------------------------------------------------------------
finished_product                         

In [0]:
# Bronze Layer — Step 15: Profile Actual Value Types
# Inspect the Python value types contained within each Bronze column.
# No source values or data types are changed.

print("BRONZE ACTUAL VALUE TYPE PROFILE")
print("=" * 130)

for block_name, df_block in bronze_delta_blocks.items():

    print(f"\nBLOCK: {block_name}")
    print("-" * 130)

    for column_name in df_block.columns:

        non_null_values = df_block[column_name].dropna()

        value_types = (
            non_null_values
            .map(lambda x: type(x).__name__)
            .value_counts()
            .to_dict()
        )

        print(
            f"{column_name:<70} | "
            f"Non-null: {len(non_null_values):<5} | "
            f"Types: {value_types}"
        )

print("\n" + "=" * 130)
print(f"Blocks profiled: {len(bronze_delta_blocks)}")
print(
    f"Columns profiled: "
    f"{sum(len(df.columns) for df in bronze_delta_blocks.values())}"
)

BRONZE ACTUAL VALUE TYPE PROFILE

BLOCK: planning_parameters
----------------------------------------------------------------------------------------------------------------------------------
parameter_name                                                         | Non-null: 7     | Types: {'str': 7}
parameter_value                                                        | Non-null: 7     | Types: {'float': 7}

BLOCK: planning_periods
----------------------------------------------------------------------------------------------------------------------------------
period                                                                 | Non-null: 8     | Types: {'int': 8}
macroperiod_month                                                      | Non-null: 8     | Types: {'int': 8}
last_day_of_period                                                     | Non-null: 8     | Types: {'int': 8}
number_of_days                                                         | Non-null: 8     | Types: {'int':

In [0]:
# Bronze Layer — Step 16: Prepare Consistent Numeric Types for Delta
# Normalize technical numeric storage only.
# No business values, codes, or meanings are changed.

integer_columns = {
    "planning_periods": [
        "period",
        "macroperiod_month",
        "last_day_of_period",
        "number_of_days"
    ],
    "product_inventory": [
        "initial_inventory_m"
    ],
    "product_successor": [
        "quantity_of_product_used_in_successor_m_m"
    ],
    "customer_order": [
        "order",
        "due_date_days_in_the_planning_horizon",
        "quantity_m"
    ],
    "demand_plan": [
        "period",
        "total_demand_m_adjusted_if_orders_are_greater_than_forecasts",
        "target_stock_m"
    ],
    "attainment_parameter": [
        "levels_of_attainment"
    ],
    "product_lot_size": [
        "minimum_lot_size_m"
    ],
    "product_production_alternative": [
        "alternative",
        "main_alternative"
    ],
    "alternative_machine": [
        "alternative",
        "machine"
    ],
    "machine": [
        "machine"
    ],
    "machine_capacity": [
        "machine",
        "period"
    ],
    "packing_machine": [
        "packing_machine"
    ],
    "product_packing_machine": [
        "packing_machine"
    ],
    "packing_capacity": [
        "packing_machine",
        "period"
    ],
    "semi_finished_acquisition": [
        "maximum_purchasing_quantity_per_macroperiod_month_m"
    ],
    "raw_material": [
        "maximum_purchasing_quantity_per_macroperiod_month_kg",
        "initial_inventory_kg"
    ]
}

float_columns = {
    "planning_parameters": [
        "parameter_value"
    ],
    "finished_product_economics": [
        "industrial_cost_eur_m",
        "margin_pct",
        "average_sales_price_eur_m"
    ],
    "semi_finished_product_cost": [
        "industrial_cost_eur_m"
    ],
    "product_raw_material_requirement": [
        "raw_material_cu_kg_m",
        "raw_material_al_kg_m",
        "raw_material_almelec_kg_m",
        "raw_material_outros_kg_m"
    ],
    "customer_order": [
        "sales_value_eur"
    ],
    "attainment_parameter": [
        "percentage_of_attainment",
        "premium_coefficient"
    ],
    "alternative_machine": [
        "production_time_sec_m"
    ],
    "machine": [
        "cost_per_unit_of_time_eur_sec"
    ],
    "machine_capacity": [
        "available_regular_time_h",
        "available_overtime_h"
    ],
    "packing_machine": [
        "cost_per_unit_of_time_eur_sec"
    ],
    "product_packing_machine": [
        "packing_time_sec_m"
    ],
    "packing_capacity": [
        "available_regular_time_h",
        "available_overtime_h"
    ],
    "semi_finished_acquisition": [
        "purchasing_cost_eur_m"
    ],
    "raw_material": [
        "purchasing_cost_eur_kg"
    ]
}

# Create a separate persistence-ready copy.
bronze_persist_blocks = {
    block_name: df.copy()
    for block_name, df in bronze_delta_blocks.items()
}

for block_name, columns in integer_columns.items():
    for column_name in columns:
        bronze_persist_blocks[block_name][column_name] = (
            pd.to_numeric(
                bronze_persist_blocks[block_name][column_name],
                errors="raise"
            ).astype("int64")
        )

for block_name, columns in float_columns.items():
    for column_name in columns:
        bronze_persist_blocks[block_name][column_name] = (
            pd.to_numeric(
                bronze_persist_blocks[block_name][column_name],
                errors="raise"
            ).astype("float64")
        )

print("BRONZE DELTA TYPE PREPARATION")
print("=" * 100)

for block_name, df_block in bronze_persist_blocks.items():
    print(
        f"{block_name:<40} | "
        f"Rows: {len(df_block):<5} | "
        f"Columns: {len(df_block.columns)}"
    )

print("=" * 100)
print(f"Blocks prepared: {len(bronze_persist_blocks)}")

BRONZE DELTA TYPE PREPARATION
planning_parameters                      | Rows: 7     | Columns: 2
planning_periods                         | Rows: 8     | Columns: 4
finished_product_type                    | Rows: 67    | Columns: 2
finished_product_economics               | Rows: 67    | Columns: 4
semi_finished_product_cost               | Rows: 225   | Columns: 2
product_inventory                        | Rows: 292   | Columns: 2
product_raw_material_requirement         | Rows: 292   | Columns: 5
product_successor                        | Rows: 1026  | Columns: 3
customer_order                           | Rows: 61    | Columns: 5
demand_plan                              | Rows: 246   | Columns: 4
attainment_parameter                     | Rows: 3     | Columns: 3
product_lot_size                         | Rows: 292   | Columns: 2
product_production_alternative           | Rows: 592   | Columns: 3
alternative_machine                      | Rows: 734   | Columns: 3
machine           

In [0]:
# Bronze Layer — Step 17: Final Pre-Write Validation
# Confirm persistence-ready Bronze blocks preserve the validated
# source structure before writing Delta tables.

print("FINAL BRONZE PRE-WRITE VALIDATION")
print("=" * 120)

prewrite_results = []

for block_name in bronze_blocks:

    source_df = bronze_blocks[block_name]
    persist_df = bronze_persist_blocks[block_name]

    expected_rows = expected_row_counts[block_name]
    actual_rows = len(persist_df)

    expected_columns = len(source_df.columns)
    actual_columns = len(persist_df.columns)

    row_status = (
        "PASS"
        if actual_rows == expected_rows
        else "FAIL"
    )

    column_status = (
        "PASS"
        if actual_columns == expected_columns
        else "FAIL"
    )

    status = (
        "PASS"
        if row_status == "PASS"
        and column_status == "PASS"
        else "FAIL"
    )

    prewrite_results.append({
        "block_name": block_name,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "expected_columns": expected_columns,
        "actual_columns": actual_columns,
        "status": status
    })

    print(
        f"{block_name:<40} | "
        f"Rows: {expected_rows} -> {actual_rows} | "
        f"Columns: {expected_columns} -> {actual_columns} | "
        f"{status}"
    )

print("=" * 120)

failed_prewrite_checks = [
    result for result in prewrite_results
    if result["status"] == "FAIL"
]

if failed_prewrite_checks:
    raise ValueError(
        f"Bronze pre-write validation failed for "
        f"{len(failed_prewrite_checks)} block(s)."
    )

print(
    f"PRE-WRITE VALIDATION PASSED: "
    f"{len(prewrite_results)} of {len(prewrite_results)} "
    f"blocks are ready for Delta persistence."
)

FINAL BRONZE PRE-WRITE VALIDATION
planning_parameters                      | Rows: 7 -> 7 | Columns: 2 -> 2 | PASS
planning_periods                         | Rows: 8 -> 8 | Columns: 4 -> 4 | PASS
finished_product_type                    | Rows: 67 -> 67 | Columns: 2 -> 2 | PASS
finished_product_economics               | Rows: 67 -> 67 | Columns: 4 -> 4 | PASS
semi_finished_product_cost               | Rows: 225 -> 225 | Columns: 2 -> 2 | PASS
product_inventory                        | Rows: 292 -> 292 | Columns: 2 -> 2 | PASS
product_raw_material_requirement         | Rows: 292 -> 292 | Columns: 5 -> 5 | PASS
product_successor                        | Rows: 1026 -> 1026 | Columns: 3 -> 3 | PASS
customer_order                           | Rows: 61 -> 61 | Columns: 5 -> 5 | PASS
demand_plan                              | Rows: 246 -> 246 | Columns: 4 -> 4 | PASS
attainment_parameter                     | Rows: 3 -> 3 | Columns: 3 -> 3 | PASS
product_lot_size                         | Rows

In [0]:
# Bronze Layer — Step 18: Persist Bronze Delta Tables
# Write the 21 validated source blocks as managed Delta tables
# in workspace.bronze_sop.

print("BRONZE DELTA TABLE PERSISTENCE")
print("=" * 110)

written_tables = []

for block_name, pandas_df in bronze_persist_blocks.items():

    table_name = f"{bronze_catalog}.{bronze_schema}.{block_name}"

    # Convert validated pandas DataFrame to Spark DataFrame
    spark_df = spark.createDataFrame(pandas_df)

    # Persist as a managed Delta table
    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    written_tables.append(table_name)

    print(
        f"{block_name:<40} | "
        f"Rows: {len(pandas_df):<5} | "
        f"WRITTEN"
    )

print("=" * 110)
print(f"Delta tables written: {len(written_tables)}")
print(f"Target schema: {bronze_catalog}.{bronze_schema}")

BRONZE DELTA TABLE PERSISTENCE
planning_parameters                      | Rows: 7     | WRITTEN
planning_periods                         | Rows: 8     | WRITTEN
finished_product_type                    | Rows: 67    | WRITTEN
finished_product_economics               | Rows: 67    | WRITTEN
semi_finished_product_cost               | Rows: 225   | WRITTEN
product_inventory                        | Rows: 292   | WRITTEN
product_raw_material_requirement         | Rows: 292   | WRITTEN
product_successor                        | Rows: 1026  | WRITTEN
customer_order                           | Rows: 61    | WRITTEN
demand_plan                              | Rows: 246   | WRITTEN
attainment_parameter                     | Rows: 3     | WRITTEN
product_lot_size                         | Rows: 292   | WRITTEN
product_production_alternative           | Rows: 592   | WRITTEN
alternative_machine                      | Rows: 734   | WRITTEN
machine                                  | Rows: 39    | WR

In [0]:
# Bronze Layer — Step 19: Verify Persisted Delta Tables
# Read each managed Delta table back from Unity Catalog
# and reconcile persisted row counts against validated source counts.

print("BRONZE DELTA PERSISTENCE VALIDATION")
print("=" * 120)

persistence_validation_results = []

for block_name, expected_rows in expected_row_counts.items():

    table_name = (
        f"{bronze_catalog}.{bronze_schema}.{block_name}"
    )

    # Confirm table exists in Unity Catalog
    table_exists = spark.catalog.tableExists(table_name)

    if table_exists:
        persisted_df = spark.table(table_name)

        actual_rows = persisted_df.count()
        actual_columns = len(persisted_df.columns)

        expected_columns = len(
            bronze_persist_blocks[block_name].columns
        )

        status = (
            "PASS"
            if actual_rows == expected_rows
            and actual_columns == expected_columns
            else "FAIL"
        )

    else:
        actual_rows = None
        actual_columns = None
        expected_columns = len(
            bronze_persist_blocks[block_name].columns
        )
        status = "FAIL"

    persistence_validation_results.append({
        "table": table_name,
        "exists": table_exists,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "expected_columns": expected_columns,
        "actual_columns": actual_columns,
        "status": status
    })

    print(
        f"{block_name:<40} | "
        f"Exists: {str(table_exists):<5} | "
        f"Rows: {expected_rows} -> {actual_rows} | "
        f"Columns: {expected_columns} -> {actual_columns} | "
        f"{status}"
    )

print("=" * 120)

failed_persistence_checks = [
    result
    for result in persistence_validation_results
    if result["status"] == "FAIL"
]

if failed_persistence_checks:
    raise ValueError(
        f"Bronze persistence validation failed for "
        f"{len(failed_persistence_checks)} table(s)."
    )

print(
    f"PERSISTENCE VALIDATION PASSED: "
    f"{len(persistence_validation_results)} of "
    f"{len(persistence_validation_results)} Delta tables "
    f"exist and reconcile successfully."
)

BRONZE DELTA PERSISTENCE VALIDATION
planning_parameters                      | Exists: True  | Rows: 7 -> 7 | Columns: 2 -> 2 | PASS
planning_periods                         | Exists: True  | Rows: 8 -> 8 | Columns: 4 -> 4 | PASS
finished_product_type                    | Exists: True  | Rows: 67 -> 67 | Columns: 2 -> 2 | PASS
finished_product_economics               | Exists: True  | Rows: 67 -> 67 | Columns: 4 -> 4 | PASS
semi_finished_product_cost               | Exists: True  | Rows: 225 -> 225 | Columns: 2 -> 2 | PASS
product_inventory                        | Exists: True  | Rows: 292 -> 292 | Columns: 2 -> 2 | PASS
product_raw_material_requirement         | Exists: True  | Rows: 292 -> 292 | Columns: 5 -> 5 | PASS
product_successor                        | Exists: True  | Rows: 1026 -> 1026 | Columns: 3 -> 3 | PASS
customer_order                           | Exists: True  | Rows: 61 -> 61 | Columns: 5 -> 5 | PASS
demand_plan                              | Exists: True  | Rows: 24

In [0]:
# Bronze Layer — Step 20: Bronze Ingestion Completion Summary
# Produce a final audit summary for the completed Bronze ingestion pipeline.

total_persisted_rows = 0

for result in persistence_validation_results:
    total_persisted_rows += result["actual_rows"]

print("PROJECT #4 — BRONZE INGESTION COMPLETION SUMMARY")
print("=" * 100)

print(f"Source file:              {source_file}")
print(f"Source worksheets:        {len(sheet_names)}")
print(f"Verified logical blocks:  {len(bronze_blocks)}")
print(f"Bronze Delta tables:      {len(persistence_validation_results)}")
print(f"Source column mappings:   {len(column_lineage)}")
print(f"Total persisted rows:     {total_persisted_rows}")
print(f"Bronze target schema:     {bronze_catalog}.{bronze_schema}")

passed_tables = sum(
    1
    for result in persistence_validation_results
    if result["status"] == "PASS"
)

print(f"Validated Delta tables:   {passed_tables}/21")

print("=" * 100)

if passed_tables != 21:
    raise ValueError(
        "Bronze ingestion completion check failed."
    )

print("BRONZE INGESTION STATUS: COMPLETE")
print()
print(
    "Raw Excel source preserved, 21 verified source blocks "
    "persisted as managed Delta tables, and all persisted "
    "tables reconciled successfully."
)

PROJECT #4 — BRONZE INGESTION COMPLETION SUMMARY
Source file:              /Volumes/workspace/default/raw_sop_source/Instance data.xlsx
Source worksheets:        7
Verified logical blocks:  21
Bronze Delta tables:      21
Source column mappings:   66
Total persisted rows:     4559
Bronze target schema:     workspace.bronze_sop
Validated Delta tables:   21/21
BRONZE INGESTION STATUS: COMPLETE

Raw Excel source preserved, 21 verified source blocks persisted as managed Delta tables, and all persisted tables reconciled successfully.
